# Deteccion de Phishing mediante Analisis Hibrido de URLs y Contenido Web
### Pipeline completo - CACIC 2026

Este notebook ejecuta las 6 fases del pipeline de punta a punta:

1. Recoleccion y etiquetado (PhishTank, OpenPhish, Tranco)
2. Normalizacion (HTML/URL)
3. Tokenizacion
4. Construccion de caracteristicas (lexicas, TF-IDF, hipervinculos)
5. Escalado
6. Particion aleatoria estratificada 70/15/15 + Entrenamiento XGBoost + Estudio de Ablacion

**IMPORTANTE - leer antes de correr:**
- El rastreo de paginas (`rastrear_corpus`) puede tardar HORAS si quieren miles de URLs, porque hace una request HTTP real por cada una con timeout de 30s. Arranquen con un subset chico (500-1000 URLs) para probar que todo funciona, despues escalen.
- OpenPhish/PhishTank dan ~500-2000 URLs *vigentes* en un momento dado. Para juntar un corpus grande hay que correr la Fase 1 varias veces a lo largo de varios dias y acumular en el mismo CSV (concatenar, no sobrescribir).
- Tranco hay que descargarlo manualmente una vez desde https://tranco-list.eu/ (boton 'Download CSV') y poner el archivo en `data/tranco_top1m.csv`.
- El experimento oficial usa una unica semilla (42) y no incluye evaluacion cronologica.

In [ ]:
# Setup: detecta si estamos en Colab o local y configura rutas
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Subir el ZIP desde tu computadora:
    # from google.colab import files; files.upload()
    # !unzip phishing-xgboost.zip
    BASE = "/content/phishing-xgboost"
else:
    cwd = os.path.abspath(os.getcwd())
    BASE = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd

sys.path.insert(0, os.path.join(BASE, "src"))
os.makedirs(os.path.join(BASE, "data"), exist_ok=True)
os.makedirs(os.path.join(BASE, "results"), exist_ok=True)
os.chdir(os.path.join(BASE, "src"))  # para que rutas relativas funcionen

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"BASE: {BASE}")
print(f"src en path: {os.path.join(BASE, 'src')}")

## 0. Instalacion de dependencias (correr una sola vez)

In [ ]:
!pip install -q xgboost scikit-learn pandas beautifulsoup4 pyarrow joblib matplotlib scipy requests

In [ ]:
import sys
sys.path.insert(0, '../src')

## 1. Fase 1 - Recoleccion y etiquetado

In [ ]:
from fase1_recoleccion import descargar_phishtank, descargar_openphish, cargar_tranco, rastrear_corpus
import pandas as pd
import os

df_phishtank = descargar_phishtank()
df_openphish = descargar_openphish()
# Antes de correr la siguiente linea: descargar el CSV de https://tranco-list.eu/ y guardarlo en ../data/tranco_top1m.csv
df_tranco = cargar_tranco('../data/tranco_top1m.csv', n=1000)

df_urls = pd.concat([df_phishtank, df_openphish, df_tranco], ignore_index=True)
df_urls = df_urls.drop_duplicates(subset='url').reset_index(drop=True)
print(f'Total URLs unicas: {len(df_urls)}')
print(df_urls['label'].value_counts())

### 1.1 Rastreo de paginas (descarga el HTML real)
**Atencion:** esto puede tardar bastante segun cuantas URLs tengan. Empiecen con un `.sample(n=500)` para probar.

In [ ]:
# Para una primera prueba rapida, descomenten la siguiente linea y usen una muestra chica:
# df_urls = df_urls.sample(n=500, random_state=42).reset_index(drop=True)

df_corpus = rastrear_corpus(df_urls, max_workers=20, out_path='../data/corpus_crudo.parquet')
df_corpus['label'].value_counts()

## 2-3. Normalizacion y Tokenizacion

In [ ]:
from fase2_3_normalizacion import procesar_corpus

df_crudo = pd.read_parquet('../data/corpus_crudo.parquet')
df_normalizado = procesar_corpus(df_crudo)
df_normalizado.to_parquet('../data/corpus_normalizado.parquet', index=False)
df_normalizado[['url', 'label']].head()

## 4-6. Construccion de features, escalado, particion, entrenamiento XGBoost y ablacion
Esta celda corre el experimento oficial y deja los resultados en `../results/student_paper_oficial/`:
- `tabla_ablacion.csv` -> tabla de resultados del paper
- `feature_importance.png` -> grafico de importancia de XGBoost
- `matriz_confusion.png` -> matriz de confusion del sistema hibrido
- `resumen_corpus.json` -> corpus y particiones
- `metricas_hibrido.json` y `config_modelo.json` -> metricas y protocolo

In [ ]:
from fase5_6_entrenamiento import correr_pipeline_completo

df_ablacion = correr_pipeline_completo()
df_ablacion

## 7. Visualizar resultados

In [ ]:
from IPython.display import Image, display
display(Image('../results/student_paper_oficial/feature_importance.png'))
display(Image('../results/student_paper_oficial/matriz_confusion.png'))